In [1]:
import pandas as pd
import openai
import json
import backoff
from tqdm import tqdm

## Load data

In [10]:
# Load urology annotations
cmb_all = pd.read_csv('../../data/urology-related/annotations/cmb_all - cmb_pure_actions1.csv')
cmb_inst_cvrg = pd.read_csv('../../data/urology-related/annotations/cmb_all - instrument coverage.csv')
cmb_verb_cvrg = pd.read_csv('../../data/urology-related/annotations/cmb_all - action_coverage.csv')
cmb_trgt_cvrg = pd.read_csv('../../data/urology-related/annotations/cmb_all - anatomic_coverage.csv')

# Format columns
cmb_inst_cvrg = cmb_inst_cvrg.rename(columns={'Instruments_fix': 'instruments', 'count_fix': 'count'})
cmb_verb_cvrg = cmb_verb_cvrg.rename(columns={'action_fix': 'verb', 'count_fix': 'count'})
cmb_trgt_cvrg = cmb_trgt_cvrg.rename(columns={'tissue_fix': 'target', 'count_fix': 'count'})

## Map Urology IAT to Clusters

In [20]:
from ast import literal_eval
import re

action_clusters = pd.read_csv('../data/urology-related/o3_action_clusters.csv').rename(columns={'Cluster': 'Name'})
instrument_clusters = pd.read_csv('../data/urology-related/o3_instrument_clusters.csv').rename(columns={'Cluster': 'Name'})
tissue_clusters = pd.read_csv('../data/urology-related/o3_tissue_clusters.csv').rename(columns={'Cluster': 'Name'})



# def eval_document(df):
#     df['Document'] = df['Document'].apply(lambda x: None if x is not None and 'NONE' in x else x)
#     df['Document'] = df['Document'].apply(lambda x: re.sub(r'\["*', '["', x) if isinstance(x, str) else x)
#     df['Document'] = df['Document'].apply(lambda x: re.sub(r'"*\]', '"]', x) if isinstance(x, str) else x)
#     df['Document'] = df['Document'].apply(lambda x: x.replace(',', '","') if isinstance(x, str) else x)
#     return df

# action_clusters = eval_document(action_clusters)
# instrument_clusters = eval_document(instrument_clusters)
# tissue_clusters = eval_document(tissue_clusters)

# def eval_strings(df, cols):
#     for col in cols:
#         df[col] = df[col].apply(lambda x: eval(x) if isinstance(x, str) else x)
#     return df

# eval_cols = ['Document', 'Representation', 'Representative_Docs']
# action_clusters = eval_strings(action_clusters, eval_cols)
# instrument_clusters = eval_strings(instrument_clusters, eval_cols)
# tissue_clusters = eval_strings(tissue_clusters, eval_cols)

# print(f"Before expansion: len(action_clusters):       {len(action_clusters)}")
# print(f"Before expansion: len(instrument_clusters):   {len(instrument_clusters)}")
# print(f"Before expansion: len(tissue_clusters):       {len(tissue_clusters)}")

# def expand_document(df):
#     # First make sure Document is properly evaluated as a list
#     df['Document'] = df['Document'].apply(lambda x: x if isinstance(x, list) else [x])

#     # Then explode the Document column
#     expanded_df = df.explode('Document')

#     # Reset the index for the new expanded dataframe
#     expanded_df = expanded_df.reset_index(drop=True)

#     return expanded_df

# action_clusters = expand_document(action_clusters)
# instrument_clusters = expand_document(instrument_clusters)
# tissue_clusters = expand_document(tissue_clusters)

# print(f"After expansion: len(action_clusters):        {len(action_clusters)}")
# print(f"After expansion: len(instrument_clusters):    {len(instrument_clusters)}")
# print(f"After expansion: len(tissue_clusters):        {len(tissue_clusters)}")

In [21]:
action_clusters

,Document,Name
0,[sweep off],perform_sweeping_maneuvers
1,[sweep],perform_sweeping_maneuvers
2,[sweep laterally],perform_sweeping_maneuvers
3,[don't sweep],perform_sweeping_maneuvers
4,[do shorter sweeps],perform_sweeping_maneuvers
...,...,...
559,[turn out],turn
560,[come out],navigate_space
561,"[come out, come out, throw back through]",navigate_space
562,"[go down, take off]",dissect


## Get Valid Annotations

In [22]:
instrument_mappings = instrument_clusters.set_index('Document')['Name'].to_dict()
action_mappings = action_clusters.set_index('Document')['Name'].to_dict()
tissue_mappings = tissue_clusters.set_index('Document')['Name'].to_dict()

In [27]:
annotations_df = cmb_all.copy()
annotations_df['instrument-cluster'] = annotations_df['instrument'].map(instrument_mappings)
annotations_df['action-cluster'] = annotations_df['action'].map(action_mappings)
annotations_df['tissue-cluster'] = annotations_df['tissue'].map(tissue_mappings)

annotations_df['instrument-cluster'] = annotations_df['instrument-cluster'].apply(lambda x: x if x != 'NONE' else None)
annotations_df['action-cluster'] = annotations_df['action-cluster'].apply(lambda x: x if x != 'NONE' else None)
annotations_df['tissue-cluster'] = annotations_df['tissue-cluster'].apply(lambda x: x if x != 'NONE' else None)

In [28]:
len(annotations_df.dropna(subset='instrument-cluster')), len(annotations_df.dropna(subset='action-cluster')), len(annotations_df.dropna(subset='tissue-cluster'))

(564, 1013, 699)

In [29]:
annotations_df.dropna(subset='instrument-cluster')['instrument-cluster'].value_counts()

instrument-cluster
left_hand             98
energy_device         81
fourth_arm            71
stitch                43
needle                40
suture_material       29
clip                  27
scissors              27
retractor             26
right_hand            19
graspers              18
catheter              17
camera                16
suction               11
general_instrument    10
bipolar               10
stapler                9
needle_driver          6
spreader               4
balloon                1
arm_instrument         1
Name: count, dtype: int64

In [30]:
annotations_df.dropna(subset='action-cluster')['action-cluster'].value_counts()

action-cluster
grasp                         146
coagulate                      87
open                           81
perform_sweeping_maneuvers     69
navigate_space                 57
ensure_safety                  57
apply_traction                 54
control_depth                  52
stop                           50
adjust_direction               44
turn                           42
inspect                        39
cut                            36
release_instrument             33
continue_action                31
spread                         25
adjust                         25
dissect                        24
suture                         14
rotate                          9
maintain_position               7
pause                           7
lift                            5
hold                            5
divide                          4
expose                          2
staple                          2
tie                             2
retract                         2

In [31]:
annotations_df.dropna(subset='tissue-cluster')['tissue-cluster'].value_counts()

tissue-cluster
ureter_bladder_urethra        162
vascular_and_major_vessels    142
prostate_tissue                92
miscellaneous                  82
seminal_vesicle_vas            49
gi_tract                       40
adrenal_kidney_region          29
muscle_capsule                 28
peritoneum_abdominal_wall      21
fat_fascia_planes              20
nerve_neurovascular            17
skeletal_landmark              12
lymph_node                      5
Name: count, dtype: int64

In [32]:
annotations_df.to_csv('../data/urology-related/annotations/cmb_all-o3_cluster_mappings.csv')